# Image Classification with Machine Learning (Advanced)

**Series:** NDVI and Remote Sensing (notebook 3 of 3)
**Time:** about 40 minutes

**In this notebook you will:**
- Make a true color image from three bands
- Let K-Means find land cover groups on its own (unsupervised)
- Train a Random Forest from labeled examples (supervised)
- Understand the difference between the two approaches

**You should already know:** notebooks 06a and 06b, and Spatial Clustering notebook 05c (K-Means).

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## Two kinds of classification

| | Unsupervised | Supervised |
|---|---|---|
| You provide | Only the number of groups | Example pixels with known labels |
| Algorithm | K-Means | Random Forest |
| Output | Groups you must name afterwards | Your own classes |

We read the bands at **one quarter resolution** (120 m pixels) so everything runs quickly in Colab.

## Step 1: Read four bands straight from GitHub

In [ ]:
!pip install -q rasterio

LANDSAT = "https://raw.githubusercontent.com/SpatialTurn/DataAnalysis/main/data/landsat"   # band2 = blue, band3 = green, band4 = red, band5 = NIR

import numpy as np
import matplotlib.pyplot as plt
import rasterio

In [ ]:
def read_band(url, scale=4):
    with rasterio.open(url) as src:
        shape = (src.height // scale, src.width // scale)
        band = src.read(1, out_shape=shape).astype("float32")
    band[band == 0] = np.nan
    return np.clip(band * 0.0000275 - 0.2, 0, 1)   # to surface reflectance (see 06a)

blue, green, red, nir = [read_band(f"{LANDSAT}/band{b}.TIF") for b in (2, 3, 4, 5)]
ndvi = (nir - red) / (nir + red)
print("Image size:", red.shape)

## Step 2: A true color image

Stacking red, green and blue gives a picture like a photo. We stretch each band between its
2nd and 98th percentile so it is not too dark.

In [ ]:
def stretch(band):
    lo, hi = np.nanpercentile(band, (2, 98))
    return np.clip((band - lo) / (hi - lo), 0, 1)

rgb = np.dstack([stretch(red), stretch(green), stretch(blue)])
rgb = np.nan_to_num(rgb, nan=1.0)    # show empty edges as white

plt.figure(figsize=(9, 8))
plt.imshow(rgb)
plt.title("True color (bands 4, 3, 2)")
plt.axis("off")
plt.show()

## Step 3: Prepare a table of pixels

Machine learning tools want a table: one **row per pixel** and one **column per feature**
(here red, green, NIR and NDVI). We drop the empty pixels.

In [ ]:
features = np.dstack([red, green, nir, ndvi])          # rows x cols x 4
table = features.reshape(-1, 4)                         # pixels x 4
valid = ~np.isnan(table).any(axis=1)
X = table[valid]
print(X.shape[0], "valid pixels,", X.shape[1], "features")

## Step 4: Unsupervised classification with K-Means

We fit K-Means on a random sample of 50,000 pixels (fast), then label every pixel.
Features are scaled first so that each one has the same spread and none dominates the distance.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

scaler = StandardScaler().fit(X)
Xs = scaler.transform(X)

rng = np.random.default_rng(0)
sample = Xs[rng.choice(len(Xs), 50_000, replace=False)]
km = KMeans(n_clusters=5, n_init=10, random_state=0).fit(sample)

labels = np.full(table.shape[0], np.nan)
labels[valid] = km.predict(Xs)
kmeans_map = labels.reshape(red.shape)

plt.figure(figsize=(9, 8))
plt.imshow(kmeans_map, cmap="tab10", vmin=0, vmax=9, interpolation="nearest")
plt.title("K-Means: 5 groups (colors have no meaning yet)")
plt.axis("off")
plt.show()

K-Means found five groups, but it does not know what they are. We name them by looking at each
group's average NDVI:

In [ ]:
for k in range(5):
    mean_ndvi = np.nanmean(ndvi[kmeans_map == k])
    print(f"Group {k}: mean NDVI {mean_ndvi:.2f}")

## Step 5: Supervised classification with Random Forest

For supervised classification we need **training data**: pixels where we know the answer.

> **In a real project** you draw training polygons in QGIS (for example 20 to 50 small polygons
> each of water, urban, forest and cropland), save them as GeoJSON, and use
> `rasterio.features.rasterize` to find their pixels.

To keep this notebook self contained, we create stand in labels from clear NDVI cases only
(very low, middle, very high). This shows the mechanics; real training polygons will give a much
better map.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

v_ndvi = X[:, 3]
y = np.full(len(X), -1)
y[v_ndvi < 0.0] = 0                           # water
y[(v_ndvi > 0.10) & (v_ndvi < 0.20)] = 1      # bare / built up
y[v_ndvi > 0.45] = 2                          # vegetation
labeled = np.where(y >= 0)[0]
pick = rng.choice(labeled, min(30_000, len(labeled)), replace=False)

X_train, X_test, y_train, y_test = train_test_split(
    X[pick][:, :3], y[pick], test_size=0.3, random_state=0)   # train on bands only, not NDVI

rf = RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=0).fit(X_train, y_train)
print(classification_report(y_test, rf.predict(X_test),
                            target_names=["water", "bare/built", "vegetation"]))

The report shows **precision** (how often a predicted class is right) and **recall** (how many real
examples of a class were found). We trained on the raw bands only, so the model had to learn the
pattern rather than copy the NDVI rule.

**Why is the score almost perfect?** Our stand in labels came from clear cut NDVI rules, which are
easy to learn. With real, hand drawn training polygons, 80 to 90% accuracy is a good result.

## Step 6: Classify the whole image

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

pred = np.full(table.shape[0], np.nan)
pred[valid] = rf.predict(X[:, :3])
rf_map = pred.reshape(red.shape)

colors = ["#2b83ba", "#d7a26b", "#1a9641"]
names = ["Water", "Bare or built up", "Vegetation"]
plt.figure(figsize=(9, 8))
plt.imshow(rf_map, cmap=ListedColormap(colors), vmin=0, vmax=2, interpolation="nearest")
plt.legend(handles=[Patch(color=c, label=n) for c, n in zip(colors, names)], loc="lower left")
plt.title("Random Forest classification")
plt.axis("off")
plt.show()

## Try it yourself

Run K-Means with **8** groups instead of 5. Do any new groups match something you can see in the true color image (rivers, cities, forest)?

<details>
<summary><b>Show a solution</b></summary>

```python
km8 = KMeans(n_clusters=8, n_init=10, random_state=0).fit(sample)
labels8 = np.full(table.shape[0], np.nan)
labels8[valid] = km8.predict(Xs)
plt.imshow(labels8.reshape(red.shape), cmap="tab10", interpolation="nearest")
plt.show()
```

</details>

In [ ]:
# Write your answer here

## Recap

- Unsupervised (K-Means) finds groups on its own; you name them afterwards.
- Supervised (Random Forest) learns your classes from labeled training pixels.
- Real training data comes from carefully drawn polygons, often made in QGIS.
- Reading rasters at lower resolution keeps machine learning fast while you learn.